# Chapter 11: Network Defense and Hardening

> "A firewall does not protect you from threats that originate inside it, threats that go around it,
> or threats that walk through its front door." paraphrased from common security teaching

---

## Learning Objectives

After completing this chapter, you will be able to:

1. Describe the role of firewalls, their types, and how to write effective rules.
2. Explain network segmentation and the DMZ architecture.
3. Describe zero-trust architecture and its practical implementation.
4. Configure basic firewall rule sets and explain the implicit deny principle.
5. Explain VPN types and their security properties.
6. Describe DNS security controls including DNSSEC, DoH, and DNS sinkholing.
7. Explain network access control (NAC) and 802.1X authentication.
8. Describe DDoS categories and mitigation strategies.

## Key Terms

- **Firewall**: a device or software that enforces access control between networks.
- **Stateful inspection**: tracks connection state to validate return traffic.
- **NGFW**: Next-Generation Firewall; adds deep packet inspection, application awareness, and IPS.
- **DMZ**: Demilitarised Zone; a network segment between the internet and the internal network.
- **Implicit deny**: any traffic not explicitly permitted is blocked.
- **ACL**: Access Control List; ordered list of permit/deny rules.
- **Zero trust**: security model requiring verification for every request regardless of network location.
- **NAC**: Network Access Control; enforces endpoint compliance before granting network access.
- **802.1X**: IEEE standard for port-based network access control.
- **Sinkhole**: redirecting malicious domain resolutions to a controlled IP for analysis or blocking.
- **Anycast**: routing technique that sends traffic to the nearest of multiple identical endpoints.

---

## Firewalls

### Firewall Types and Evolution

#### Packet-Filter Firewalls

Packet-filter firewalls evaluate each packet independently against a rule set based on header fields:
source/destination IP, port, and protocol. They are fast and simple but cannot track connection
state, making them vulnerable to IP spoofing and fragmentation attacks. They operate at layer 3-4.

#### Stateful Inspection Firewalls

Stateful firewalls maintain a connection table that tracks TCP sessions and UDP pseudo-sessions.
Return traffic is automatically permitted if it belongs to an established, allowed session. This
prevents certain spoofing attacks and simplifies outbound rule writing. The state table itself is
a resource that can be exhausted by SYN floods.

#### Next-Generation Firewalls

NGFWs add layer-7 inspection: application identification (blocking BitTorrent regardless of port),
user identity-based rules (allow finance team to access financial SaaS), URL filtering, TLS
inspection (decrypt-inspect-re-encrypt), and integrated intrusion prevention. NGFWs are the
standard for enterprise perimeter and internal segmentation today.

### Writing Firewall Rules

#### The Implicit Deny Principle

All enterprise firewall rule sets should end with an explicit `deny all` rule (or rely on an
implicit deny that blocks everything not matched above). Traffic is permitted only by explicit
positive rules. This default-deny posture means new services are blocked until consciously allowed,
rather than exposed until someone notices.

#### Rule Ordering and Specificity

Firewall rules are evaluated top to bottom; the first match wins. More specific rules must precede
more general ones. A common misconfiguration is placing a broad permit rule before a more specific
deny, inadvertently allowing everything that the deny was meant to block.

---

## Network Segmentation

### DMZ Architecture

A DMZ (Demilitarised Zone) places internet-facing servers (web, mail, DNS) in a network segment
that is reachable from the internet but cannot initiate connections to the internal network. The
DMZ sits between two firewalls: the outer firewall permits inbound traffic to DMZ services; the
inner firewall permits only specific, authorised traffic from the DMZ to the internal network
(e.g., the web server may query the internal database on port 5432 only). If the web server is
compromised, the attacker cannot reach the internal network without breaching the inner firewall.

### VLAN and Micro-Segmentation

VLANs create logical network segments that limit broadcast domains and create enforcement points.
Micro-segmentation extends this to the workload level: even hosts on the same VLAN have explicit
rules about which port/protocol combinations can communicate. East-west traffic (between internal
workloads) is filtered just as rigorously as north-south traffic (to/from internet). Micro-
segmentation dramatically limits lateral movement after initial compromise.

---

## Zero-Trust Architecture

### The Zero-Trust Principle

Zero trust abandons the assumption that everything inside the network perimeter is safe. Every
access request is evaluated against policy regardless of where it originates: inside the office,
on VPN, or from a cloud workload. The three core principles are:

1. Verify explicitly: authenticate and authorise every request using all available data points
   (identity, device health, location, time, data sensitivity).
2. Use least-privilege access: grant only the permissions needed for the specific request.
3. Assume breach: minimise blast radius through segmentation; assume an attacker is already inside
   and design accordingly.

#### Identity-Centric Access

In a zero-trust model, the network is not the trust boundary; the identity is. A user with strong
MFA and a healthy, compliant device gets access. The same user on an unmanaged device gets reduced
access or is denied. A compromised account, even from inside the office, cannot reach sensitive
resources without the correct device posture and MFA.

---

## DNS Security

### DNSSEC

DNSSEC adds cryptographic signatures to DNS resource records. A DNSSEC-aware resolver validates
the chain of signatures from the root zone down to the answer, detecting tampered or forged records.
DNSSEC prevents cache poisoning but does not encrypt DNS traffic (queries and responses remain
visible).

### DNS over HTTPS and DNS over TLS

DoH and DoT encrypt the DNS channel, preventing eavesdropping on queries. DoH sends DNS queries
inside HTTPS traffic on port 443, making it indistinguishable from normal web traffic. DoT uses
a dedicated port (853) and allows enterprise filtering. Enterprise deployments often force DoT to
a corporate resolver that applies sinkholing and filtering policies.

### DNS Sinkholing

A sinkhole redirects DNS queries for known-malicious domains to a controlled IP address rather
than the actual C2 server. Malware that checks in with a sinkholed domain is identified (the
connecting host is infected) and prevented from reaching its real C2. Threat intelligence feeds
supply the domain blocklists; enterprise DNS resolvers apply them automatically.

---

## VPNs and Remote Access

### IPsec and WireGuard

IPsec provides network-layer encryption and authentication, operating in tunnel mode (encrypting
the entire IP packet, including headers) for site-to-site VPNs or transport mode for host-to-host.
WireGuard is a modern, lean VPN protocol that uses state-of-the-art cryptography (Curve25519,
ChaCha20, Poly1305) with a minimal code base, making it faster to audit and in practice faster than
IPsec or OpenVPN.

### Split Tunnelling and Its Risks

Split tunnelling routes only corporate-destined traffic through the VPN while internet traffic goes
directly to the user's ISP. This reduces latency and VPN gateway load but means the VPN provides no
visibility or control over the user's general internet activity. A user infected with malware while
on split-tunnel VPN may continue to communicate with C2 without the corporate security stack seeing
it.

---

## Network Access Control and 802.1X

NAC enforces endpoint health before granting network access. Before a device is permitted onto the
network it must present valid credentials (802.1X) and pass a health check (antivirus current,
OS patches applied, disk encryption enabled). Non-compliant devices are quarantined to a remediation
VLAN with access only to update servers.

### 802.1X Operation

In 802.1X, three components interact: the supplicant (client device), the authenticator (switch or
wireless access point), and the authentication server (RADIUS). The switch blocks all traffic from
a newly connected device except EAP (Extensible Authentication Protocol) frames until the
authentication server validates the device's credentials. Only then does the switch permit normal
traffic.

---

## DDoS and Mitigation

### DDoS Attack Categories

| Category | Mechanism | Example |
|---|---|---|
| Volumetric | Saturate bandwidth | DNS amplification, UDP flood |
| Protocol | Exhaust state tables | SYN flood, fragmentation |
| Application-layer | Exhaust server resources | HTTP flood, Slowloris |

### DDoS Mitigation

Cloud-based scrubbing services (Cloudflare, Akamai, AWS Shield) absorb attack traffic using
anycast routing, which distributes the attack across a global network. Rate limiting at the
network edge drops traffic from sources exceeding defined thresholds. BGP blackholing redirects
attack traffic to null routes. Application-layer DDoS mitigation requires distinguishing legitimate
from bot traffic using CAPTCHA, JavaScript challenges, and browser fingerprinting.

---

## Why This Matters

Network defense is the layer that contains damage once a host is compromised. A firewall that limits
outbound traffic prevents data exfiltration. A DMZ limits blast radius when a web server is
compromised. A NAC-enforced network prevents an unpatched guest device from spreading malware to
corporate assets. Zero-trust architecture means a compromised credential does not automatically
grant access to every resource. These controls work together to shorten the attacker's kill chain.

---

## News in Focus

Several nation-state intrusion campaigns achieved network-wide compromise specifically because
east-west traffic was unfiltered: once the attacker gained a foothold in one workload, they could
reach any other workload on the same flat network using standard protocols (SMB, RDP, WMI). The
entire subsequent kill chain depended on the absence of micro-segmentation. Post-incident
recommendations universally included implementing east-west filtering and zero-trust principles
for workload communication.

---


In [1]:
# Chapter 11 -- Firewall rule evaluator and network segment model

from dataclasses import dataclass
from typing import Optional

@dataclass
class FirewallRule:
    action: str           # PERMIT or DENY
    src_ip: str           # CIDR or "any"
    dst_ip: str           # CIDR or "any"
    protocol: str         # tcp, udp, icmp, any
    dst_port: Optional[int]  # None for "any"
    description: str

def ip_in_cidr(ip, cidr):
    if cidr == "any":
        return True
    if "/" not in cidr:
        return ip == cidr
    base, prefix = cidr.split("/")
    prefix = int(prefix)
    def to_int(addr):
        parts = list(map(int, addr.split(".")))
        return (parts[0]<<24)|(parts[1]<<16)|(parts[2]<<8)|parts[3]
    mask = (0xFFFFFFFF << (32-prefix)) & 0xFFFFFFFF
    return (to_int(ip) & mask) == (to_int(base) & mask)

def evaluate(rules, pkt):
    for i, r in enumerate(rules):
        if not ip_in_cidr(pkt["src"], r.src_ip):
            continue
        if not ip_in_cidr(pkt["dst"], r.dst_ip):
            continue
        if r.protocol != "any" and r.protocol != pkt["proto"]:
            continue
        if r.dst_port is not None and r.dst_port != pkt["port"]:
            continue
        return r.action, i+1, r.description
    return "DENY", 0, "Implicit deny (no rule matched)"

# DMZ firewall rule set
rules = [
    FirewallRule("PERMIT","0.0.0.0/0","203.0.113.10","tcp",443,"Allow HTTPS to web server"),
    FirewallRule("PERMIT","0.0.0.0/0","203.0.113.10","tcp",80, "Allow HTTP to web server"),
    FirewallRule("PERMIT","203.0.113.10","10.0.0.50","tcp",5432,"Web -> DB on port 5432 only"),
    FirewallRule("DENY",  "203.0.113.10","10.0.0.0/8","any",None,"Block DMZ to internal except DB"),
    FirewallRule("PERMIT","10.0.0.0/8","any","any",None,"Allow internal outbound"),
    FirewallRule("DENY",  "any","any","any",None,"Explicit deny all"),
]

packets = [
    dict(src="1.2.3.4",      dst="203.0.113.10", proto="tcp", port=443, label="Internet -> Web HTTPS"),
    dict(src="203.0.113.10", dst="10.0.0.50",    proto="tcp", port=5432,label="Web -> DB query"),
    dict(src="203.0.113.10", dst="10.0.0.1",     proto="tcp", port=22,  label="Web -> Internal SSH (lateral)"),
    dict(src="10.0.0.5",     dst="8.8.8.8",      proto="udp", port=53,  label="Internal -> Internet DNS"),
    dict(src="5.5.5.5",      dst="10.0.0.1",     proto="tcp", port=3389,label="Internet -> Internal RDP (attack)"),
]

print(f"{'Packet':<40} {'Action':<8} {'Rule#':<7} {'Note'}")
print("-" * 90)
for p in packets:
    action, rnum, desc = evaluate(rules, p)
    print(f"{p['label']:<40} {action:<8} {rnum:<7} {desc}")


Packet                                   Action   Rule#   Note
------------------------------------------------------------------------------------------
Internet -> Web HTTPS                    PERMIT   1       Allow HTTPS to web server
Web -> DB query                          PERMIT   3       Web -> DB on port 5432 only
Web -> Internal SSH (lateral)            DENY     4       Block DMZ to internal except DB
Internal -> Internet DNS                 PERMIT   5       Allow internal outbound
Internet -> Internal RDP (attack)        DENY     6       Explicit deny all


## Review Questions (MCQ)

**Q1.** The implicit deny principle means:
A. All traffic is denied until a user logs in  B. Any traffic not explicitly permitted by a rule is blocked  C. Deny rules take precedence over permit rules  D. All outbound traffic is denied by default

**Q2.** A DMZ is placed between:
A. Two internal switches  B. The internet-facing and internal-facing firewalls  C. The VPN and the corporate network  D. The RADIUS server and the switch

**Q3.** In zero-trust architecture, the primary trust boundary is:
A. The firewall  B. The corporate office network  C. Verified identity with device posture  D. The VPN

**Q4.** DNSSEC protects against:
A. DNS query eavesdropping  B. Cache poisoning by signing resource records  C. DDoS against name servers  D. Subdomain takeover

**Q5.** A DNS sinkhole primarily helps defenders:
A. Speed up DNS resolution  B. Identify infected hosts attempting to reach malicious C2 domains  C. Prevent DNSSEC failures  D. Enforce DoT

**Q6.** Split-tunnel VPN poses a security risk because:
A. It is slower than full-tunnel  B. Internet traffic bypasses corporate visibility and security controls  C. It requires 802.1X  D. It uses insecure protocols

**Q7.** In 802.1X, the switch acts as the:
A. Supplicant  B. Authentication server  C. Authenticator  D. Certificate Authority

**Q8.** A SYN flood attack targets which resource?
A. Disk I/O  B. The server's TCP connection-state table  C. SSL certificate validity  D. DNS cache

**Q9.** Anycast routing is used in DDoS mitigation to:
A. Encrypt attack traffic  B. Distribute attack traffic across a global network of scrubbing nodes  C. Block UDP amplification  D. Rate-limit HTTP requests

**Q10.** Micro-segmentation primarily limits:
A. Internet-facing attack surface  B. East-west (lateral) movement within the network  C. DDoS impact  D. DNS poisoning

*Answers: Q1 B, Q2 B, Q3 C, Q4 B, Q5 B, Q6 B, Q7 C, Q8 B, Q9 B, Q10 B.*

## Lab Assignment

**Part A -- Firewall rule audit**: Using the evaluator above, add three additional test packets that expose a gap or misconfiguration in the existing rule set. For each packet, explain what attack scenario it represents and what rule change would prevent it.

**Part B -- DMZ design**: Draw (or describe in table form) a three-tier DMZ architecture for a company that hosts a public web application, an internal HR system, and a database server. Specify the firewall rules between each zone.

**Part C -- Zero-trust gap analysis**: For an organisation that currently uses a traditional VPN model, identify five specific changes needed to implement zero-trust principles. For each, specify the technology or process change and the risk it addresses.

**Part D -- DNS security audit**: Run `dig +dnssec example.com` and `dig +dnssec google.com`. Check whether DNSSEC is enabled (look for RRSIG records). Then check whether your local resolver enforces DNSSEC validation. Document your findings and any differences between sites.

## References

```{bibliography}
:filter: docname in docnames
```


```{index} Firewall, Stateful inspection, NGFW, DMZ, Implicit deny, ACL, Zero trust, NAC, 802.1X, Sinkhole, Anycast
```
